# Notebook 03 — Model Definition: AVXAIModel

**Project:** AV-XAI (Explainable Multimodal Deepfake Detection)
**Goal:** Define the full model architecture: ViViT-B/16 + WavLM-Large + cross-modal consistency + hierarchical fusion. Smoke-test a forward pass on real preprocessed clips. Save as a reusable Python module for nb04 (training).

## Architecture (per project_log §2)

```
INPUT:
  faces  [B, 16, 3, 224, 224] uint8  →  normalize [-1, 1] (ViViT stats)
  audio  [B, 160000] float32         →  per-instance zero-mean unit-variance (WavLM convention)

VIDEO BRANCH:
  ViViT-B/16 (Kinetics-400 pretrained, full fine-tune)
    last_hidden_state[:, 0]  →  vt [B, 768]   (CLS token)

AUDIO BRANCH:
  WavLM-Large (LibriSpeech pretrained, full fine-tune)
    mean(last_hidden_state, dim=1)  →  at [B, 1024]

CONSISTENCY HEAD (for contrastive loss):
  vt → Linear(768→256) → L2-norm  →  v_embed
  at → Linear(1024→256) → L2-norm →  a_embed

HIERARCHICAL FUSION:
  Level 1 — Feature:
    Project at: Linear(1024→768)
    Stack [vt, at_proj] → [B, 2, 768]
    Transformer encoder (2 layers, 4 heads, d_model=768, ffn=3072)
    Mean-pool over 2 tokens → fused [B, 768]
  Level 2 — Decision (per-modality classifiers):
    video_logits = Linear(768→4)(vt)
    audio_logits = Linear(1024→4)(at)
    fused_logits = Linear(768→4)(fused)
  Level 3 — Ensemble (soft vote):
    ensemble_probs = mean(softmax(video), softmax(audio), softmax(fused))
    logits_4class = log(ensemble_probs + ε)

BINARY HEAD (secondary):
  logits_binary = Linear(768→2)(fused)

OUTPUT DICT:
  {
    'logits_4class':  [B, 4]   ensemble (primary target)
    'logits_binary':  [B, 2]
    'logits_fused':   [B, 4]   feature-level only
    'logits_video':   [B, 4]   video-only
    'logits_audio':   [B, 4]   audio-only
    'v_embed':        [B, 256]  for contrastive loss
    'a_embed':        [B, 256]  for contrastive loss
  }
```

## Notebook structure

- **Phase A** — Load ViViT-B/16, smoke test forward
- **Phase B** — Load WavLM-Large, smoke test forward
- **Phase C** — Consistency projection module
- **Phase D** — Hierarchical fusion module
- **Phase E** — Assemble full `AVXAIModel` class
- **Phase F** — Forward pass on 2 real preprocessed clips (one real, one fake)
- **Phase G** — Loss functions: `detection_loss`, `contrastive_loss`, `total_loss`
- **Phase H** — Save `avxai_model.py` to `models/` for nb04 import

## Sanity targets

- ViViT-B/16 ~86M params
- WavLM-Large ~316M params
- Fusion + heads ~3M params
- **Total ~405M params, all trainable (full fine-tune)**
- VRAM at micro-batch 4, bf16, gradient checkpointing: ~38 GB (fits A6000 48 GB)

In [ ]:
# --- Imports and paths ---
import os, sys, json, time, math
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# HuggingFace
from transformers import VivitConfig, VivitModel
from transformers import WavLMModel, WavLMConfig

# --- Paths ---
PROJECT_ROOT = Path(r"C:\Users\Komal-Sch\Desktop\AV-XAI")
DATA_ROOT    = PROJECT_ROOT / "data"
PREPROC_ROOT = DATA_ROOT / "preprocessed"
MODELS_DIR   = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# --- Device ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU   : {torch.cuda.get_device_name(0)}")
    print(f"VRAM  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    torch.cuda.empty_cache()

# --- Constants (must match preprocessing) ---
N_FRAMES      = 16
IMG_SIZE      = 224
AUDIO_SR      = 16_000
AUDIO_SAMPLES = AUDIO_SR * 10   # 160,000

# ViViT-B/16 Kinetics-400 normalization (NOT ImageNet stats)
VIVIT_MEAN = [0.5, 0.5, 0.5]
VIVIT_STD  = [0.5, 0.5, 0.5]

## Phase A — Load ViViT-B/16 (Kinetics-400 pretrained)

The HF checkpoint `google/vivit-b-16x2-kinetics400` was trained with `num_frames=32` and `tubelet_size=2` → 16 temporal positions. We're using 16 frames → 8 temporal positions. Loading with `ignore_mismatched_sizes=True` reinitializes the temporal positional embeddings only; spatial patches and transformer blocks load cleanly. Fine-tuning re-learns the temporal embeddings on LAV-DF.

**Note:** First run downloads ~340 MB from HuggingFace. Cached afterwards.

In [ ]:
# --- Load ViViT ---
print("Loading ViViT-B/16 (Kinetics-400)...")

vivit_config = VivitConfig.from_pretrained("google/vivit-b-16x2-kinetics400")
vivit_config.num_frames = N_FRAMES
print(f"  num_frames        : {vivit_config.num_frames}")
print(f"  image_size        : {vivit_config.image_size}")
print(f"  tubelet_size      : {vivit_config.tubelet_size}")
print(f"  hidden_size       : {vivit_config.hidden_size}")
print(f"  num_hidden_layers : {vivit_config.num_hidden_layers}")
print(f"  num_attention_heads: {vivit_config.num_attention_heads}")

vivit = VivitModel.from_pretrained(
    "google/vivit-b-16x2-kinetics400",
    config=vivit_config,
    ignore_mismatched_sizes=True,   # truncates temporal pos embeddings
)
vivit_params = sum(p.numel() for p in vivit.parameters())
print(f"\nViViT loaded. Params: {vivit_params/1e6:.1f}M")

# --- Smoke test forward pass ---
vivit.eval()
with torch.no_grad():
    # Input: [B, T, C, H, W] in [-1, 1] range
    dummy = torch.randn(1, N_FRAMES, 3, IMG_SIZE, IMG_SIZE)
    out = vivit(pixel_values=dummy)
    last_hidden = out.last_hidden_state
    cls_token   = last_hidden[:, 0]
    print(f"\nForward pass OK.")
    print(f"  last_hidden_state: {tuple(last_hidden.shape)}  (B, num_tokens, hidden)")
    print(f"  [CLS] token      : {tuple(cls_token.shape)}    (B, 768)")

## Phase B — Load WavLM-Large (LibriSpeech pretrained)

`microsoft/wavlm-large` is the 24-layer, 1024-dim self-supervised speech model. Input is raw waveform at 16 kHz. Output is `last_hidden_state` of shape `[B, T_audio, 1024]` where T_audio ≈ 499 for our 10s clips (WavLM downsamples ~320× internally).

WavLM expects per-instance zero-mean unit-variance normalization. We apply this in the wrapper, not via `Wav2Vec2FeatureExtractor`, to keep the forward pass differentiable.

**Note:** First run downloads ~1.2 GB. Cached afterwards.

In [ ]:
# --- Load WavLM ---
print("Loading WavLM-Large (LibriSpeech)...")

wavlm = WavLMModel.from_pretrained("microsoft/wavlm-large")
wavlm_params = sum(p.numel() for p in wavlm.parameters())
print(f"  hidden_size       : {wavlm.config.hidden_size}")
print(f"  num_hidden_layers : {wavlm.config.num_hidden_layers}")
print(f"  num_attention_heads: {wavlm.config.num_attention_heads}")
print(f"\nWavLM loaded. Params: {wavlm_params/1e6:.1f}M")

# --- Smoke test forward pass ---
wavlm.eval()
with torch.no_grad():
    dummy_audio = torch.randn(1, AUDIO_SAMPLES)
    # Per-instance normalize (WavLM convention)
    dummy_audio = (dummy_audio - dummy_audio.mean(dim=-1, keepdim=True)) / (
        dummy_audio.std(dim=-1, keepdim=True) + 1e-7
    )
    out = wavlm(input_values=dummy_audio)
    last_hidden = out.last_hidden_state
    pooled = last_hidden.mean(dim=1)
    print(f"\nForward pass OK.")
    print(f"  last_hidden_state: {tuple(last_hidden.shape)}  (B, T_audio≈499, 1024)")
    print(f"  mean-pooled      : {tuple(pooled.shape)}        (B, 1024)")

## Phase C — Cross-modal consistency module

Projects `vt [B, 768]` and `at [B, 1024]` into a shared 256-dim space, L2-normalizes both. The normalized embeddings are used by `contrastive_loss` (Phase G) to pull paired `(vt, at)` together and push shuffled `(vt, a't)` apart.

This is the AV-XAI paper's "cross-modal consistency" mechanism. It works as an auxiliary loss that regularizes the encoders to produce semantically aligned representations across modalities — the model learns to encode "what a fake voice sounds like" and "what a fake face looks like" in coherent ways.

In [ ]:
class ConsistencyProjector(nn.Module):
    """Projects video & audio embeddings to a shared L2-normalized space."""
    def __init__(self, d_video=768, d_audio=1024, d_shared=256):
        super().__init__()
        self.v_proj = nn.Sequential(
            nn.Linear(d_video, d_shared * 2),
            nn.GELU(),
            nn.Linear(d_shared * 2, d_shared),
        )
        self.a_proj = nn.Sequential(
            nn.Linear(d_audio, d_shared * 2),
            nn.GELU(),
            nn.Linear(d_shared * 2, d_shared),
        )

    def forward(self, vt, at):
        v = self.v_proj(vt)
        a = self.a_proj(at)
        v = F.normalize(v, dim=-1)
        a = F.normalize(a, dim=-1)
        return v, a


# Smoke test
proj = ConsistencyProjector()
vt = torch.randn(4, 768)
at = torch.randn(4, 1024)
ve, ae = proj(vt, at)
print(f"v_embed: {tuple(ve.shape)}  ||  norm: {ve.norm(dim=-1)[:2].tolist()}")
print(f"a_embed: {tuple(ae.shape)}  ||  norm: {ae.norm(dim=-1)[:2].tolist()}")
print(f"ConsistencyProjector params: {sum(p.numel() for p in proj.parameters())/1e3:.1f}K")

## Phase D — Hierarchical fusion module

Three levels:

**Level 1 — Feature-level** (cross-modal interaction):
Project audio `at` from 1024 → 768. Stack `[vt, at_proj]` as a 2-token sequence `[B, 2, 768]`. Pass through 2 layers of `nn.TransformerEncoderLayer` (4 heads, d_model=768, FFN=3072, dropout=0.1, pre-norm). The self-attention over 2 tokens IS bidirectional cross-modal cross-attention (video attends to audio + vice versa, with multi-head learnable projections). Mean-pool over the 2 output tokens to get `fused [B, 768]`.

**Level 2 — Decision-level** (per-modality heads):
Three independent 4-class classifiers from `vt`, `at`, and `fused`. These produce `logits_video`, `logits_audio`, `logits_fused`.

**Level 3 — Ensemble** (soft vote):
Average the softmax probabilities of the three decision-level outputs, then take `log()` to get the ensemble logits for the primary loss.

The architecture also exposes the three intermediate per-modality losses so we can train them jointly (matches the paper's "decision-level supervision" idea).

In [ ]:
class HierarchicalFusion(nn.Module):
    """Three-level hierarchical fusion: feature + decision + ensemble."""
    def __init__(self, d_video=768, d_audio=1024, d_model=768, num_heads=4,
                 num_layers=2, n_classes=4, dropout=0.1):
        super().__init__()
        # Level 1: align audio to d_model, then cross-modal Transformer
        self.audio_align = nn.Linear(d_audio, d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=num_heads,
            dim_feedforward=d_model * 4,    # 3072 for d_model=768
            dropout=dropout,
            batch_first=True,
            activation="gelu",
            norm_first=True,                 # pre-norm: more stable for fine-tuning
        )
        self.fusion_transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # Level 2: per-modality decision heads
        self.head_video  = nn.Linear(d_video, n_classes)
        self.head_audio  = nn.Linear(d_audio, n_classes)
        self.head_fused  = nn.Linear(d_model, n_classes)

        # Binary head off the fused vector
        self.head_binary = nn.Linear(d_model, 2)

    def forward(self, vt, at):
        # vt: [B, 768], at: [B, 1024]

        # Level 1: feature-level cross-modal interaction
        at_aligned = self.audio_align(at)                   # [B, 768]
        tokens = torch.stack([vt, at_aligned], dim=1)       # [B, 2, 768]
        tokens = self.fusion_transformer(tokens)            # [B, 2, 768]
        fused = tokens.mean(dim=1)                          # [B, 768]

        # Level 2: per-modality 4-class logits
        logits_video = self.head_video(vt)
        logits_audio = self.head_audio(at)
        logits_fused = self.head_fused(fused)

        # Level 3: ensemble soft vote (mean of softmax probs → log for stable CE)
        probs_v = F.softmax(logits_video, dim=-1)
        probs_a = F.softmax(logits_audio, dim=-1)
        probs_f = F.softmax(logits_fused, dim=-1)
        ensemble_probs = (probs_v + probs_a + probs_f) / 3.0
        logits_4class  = torch.log(ensemble_probs + 1e-9)

        # Binary head off fused
        logits_binary = self.head_binary(fused)

        return {
            "logits_4class": logits_4class,
            "logits_binary": logits_binary,
            "logits_fused":  logits_fused,
            "logits_video":  logits_video,
            "logits_audio":  logits_audio,
            "fused":         fused,
        }


# Smoke test
fusion = HierarchicalFusion()
vt = torch.randn(4, 768)
at = torch.randn(4, 1024)
out = fusion(vt, at)
for k, v in out.items():
    print(f"  {k:<16}: {tuple(v.shape)}")
print(f"\nHierarchicalFusion params: {sum(p.numel() for p in fusion.parameters())/1e6:.2f}M")

## Phase E — Full `AVXAIModel`

Combines ViViT + WavLM + ConsistencyProjector + HierarchicalFusion into one `nn.Module`. The forward pass takes raw preprocessed inputs (uint8 faces, float32 audio) and handles normalization internally so the training loop stays clean.

**Gradient checkpointing** is enabled on both backbones to fit micro-batch 4 in 48 GB VRAM with bf16. Toggle off if you want full-speed forward at higher VRAM cost.

In [ ]:
class AVXAIModel(nn.Module):
    """Full AV-XAI model: ViViT + WavLM + cross-modal consistency + hierarchical fusion."""

    def __init__(
        self,
        vivit_model_name: str = "google/vivit-b-16x2-kinetics400",
        wavlm_model_name: str = "microsoft/wavlm-large",
        num_frames: int = 16,
        d_shared: int = 256,
        n_classes: int = 4,
        fusion_layers: int = 2,
        fusion_heads: int = 4,
        use_gradient_checkpointing: bool = True,
    ):
        super().__init__()
        # --- ViViT ---
        vivit_cfg = VivitConfig.from_pretrained(vivit_model_name)
        vivit_cfg.num_frames = num_frames
        self.vivit = VivitModel.from_pretrained(
            vivit_model_name, config=vivit_cfg, ignore_mismatched_sizes=True
        )

        # --- WavLM ---
        self.wavlm = WavLMModel.from_pretrained(wavlm_model_name)

        # --- Cross-modal consistency ---
        self.consistency = ConsistencyProjector(
            d_video=self.vivit.config.hidden_size,
            d_audio=self.wavlm.config.hidden_size,
            d_shared=d_shared,
        )

        # --- Hierarchical fusion ---
        self.fusion = HierarchicalFusion(
            d_video=self.vivit.config.hidden_size,
            d_audio=self.wavlm.config.hidden_size,
            d_model=self.vivit.config.hidden_size,
            num_heads=fusion_heads,
            num_layers=fusion_layers,
            n_classes=n_classes,
        )

        # --- Normalization buffers ---
        self.register_buffer(
            "vivit_mean", torch.tensor(VIVIT_MEAN).view(1, 1, 3, 1, 1), persistent=False
        )
        self.register_buffer(
            "vivit_std", torch.tensor(VIVIT_STD).view(1, 1, 3, 1, 1), persistent=False
        )

        # --- Gradient checkpointing ---
        if use_gradient_checkpointing:
            self.vivit.gradient_checkpointing_enable()
            self.wavlm.gradient_checkpointing_enable()

    # ------------------------------------------------------------------ #
    # Input normalization (done inside the module so train loop is clean)
    # ------------------------------------------------------------------ #
    def _normalize_video(self, faces_uint8: torch.Tensor) -> torch.Tensor:
        """faces_uint8: [B, T, 3, H, W] uint8  →  float32 normalized."""
        x = faces_uint8.float() / 255.0
        x = (x - self.vivit_mean) / self.vivit_std
        return x

    def _normalize_audio(self, audio_f32: torch.Tensor) -> torch.Tensor:
        """Per-instance zero-mean unit-variance normalization."""
        mean = audio_f32.mean(dim=-1, keepdim=True)
        std  = audio_f32.std(dim=-1, keepdim=True) + 1e-7
        return (audio_f32 - mean) / std

    # ------------------------------------------------------------------ #
    # Forward
    # ------------------------------------------------------------------ #
    def forward(self, faces, audio):
        # faces: [B, 16, 3, 224, 224] uint8 (or already-normalized float)
        # audio: [B, 160000] float32

        if faces.dtype == torch.uint8:
            faces = self._normalize_video(faces)
        audio = self._normalize_audio(audio)

        # Video branch — ViViT
        vivit_out = self.vivit(pixel_values=faces)
        vt = vivit_out.last_hidden_state[:, 0]       # [B, 768] (CLS token)

        # Audio branch — WavLM
        wavlm_out = self.wavlm(input_values=audio)
        at = wavlm_out.last_hidden_state.mean(dim=1) # [B, 1024]

        # Cross-modal consistency embeddings
        v_embed, a_embed = self.consistency(vt, at)

        # Hierarchical fusion + classification
        fusion_out = self.fusion(vt, at)

        return {
            **fusion_out,
            "v_embed": v_embed,
            "a_embed": a_embed,
            "vt":      vt,
            "at":      at,
        }


print("AVXAIModel class defined.")

## Phase F — Forward pass smoke test (on real preprocessed clips)

Load 2 preprocessed `.pt` files (one real, one fake), batch them, and run a forward pass. Assert every expected output shape. Print parameter count, VRAM usage, and timing.

This is the moment of truth — if anything's wrong with how we glued ViViT and WavLM together, it shows up here.

In [ ]:
# --- Pick two real clips from disk ---
import random
random.seed(0)
train_files = sorted((PREPROC_ROOT / "train").glob("*.pt"))
test_clips = random.sample(train_files, 2)
print(f"Smoke test on:")
for fp in test_clips:
    print(f"  {fp.name}")

# --- Load and batch ---
batch_faces, batch_audio, batch_labels = [], [], []
for fp in test_clips:
    e = torch.load(fp, weights_only=False)
    batch_faces.append(e["faces"])
    batch_audio.append(e["audio"])
    batch_labels.append(e["label_4class"])

faces = torch.stack(batch_faces, dim=0)       # [2, 16, 3, 224, 224] uint8
audio = torch.stack(batch_audio, dim=0)       # [2, 160000] float32
labels = torch.tensor(batch_labels)

print(f"\nfaces : {tuple(faces.shape)} {faces.dtype}")
print(f"audio : {tuple(audio.shape)} {audio.dtype}")
print(f"labels: {labels.tolist()}")

# --- Build full model on GPU ---
print("\nBuilding AVXAIModel (this loads both pretrained checkpoints)...")
torch.cuda.empty_cache() if device.type == "cuda" else None
model = AVXAIModel(use_gradient_checkpointing=True).to(device)
model.eval()

total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal params    : {total_params/1e6:.1f}M")
print(f"Trainable params: {trainable/1e6:.1f}M")

# --- Forward pass ---
faces_dev = faces.to(device)
audio_dev = audio.to(device)

torch.cuda.synchronize() if device.type == "cuda" else None
t0 = time.time()
with torch.no_grad():
    out = model(faces_dev, audio_dev)
torch.cuda.synchronize() if device.type == "cuda" else None
elapsed = time.time() - t0

print(f"\nForward pass: {elapsed*1000:.1f} ms (batch=2, fp32, no_grad)")

# --- Assert output shapes ---
expected_shapes = {
    "logits_4class":  (2, 4),
    "logits_binary":  (2, 2),
    "logits_fused":   (2, 4),
    "logits_video":   (2, 4),
    "logits_audio":   (2, 4),
    "v_embed":        (2, 256),
    "a_embed":        (2, 256),
    "vt":             (2, 768),
    "at":             (2, 1024),
    "fused":          (2, 768),
}
print("\nOutput shapes:")
for k, expected in expected_shapes.items():
    actual = tuple(out[k].shape)
    flag = "OK" if actual == expected else "!! WRONG"
    print(f"  {k:<16}: {actual!s:<25} expected {expected!s:<12} [{flag}]")
    assert actual == expected, f"Shape mismatch on {k}"

# --- L2-norm check for consistency embeddings ---
v_norms = out["v_embed"].norm(dim=-1)
a_norms = out["a_embed"].norm(dim=-1)
print(f"\nv_embed L2 norms: {v_norms.tolist()}  (expect ~1.0)")
print(f"a_embed L2 norms: {a_norms.tolist()}  (expect ~1.0)")
assert (v_norms - 1.0).abs().max() < 1e-4
assert (a_norms - 1.0).abs().max() < 1e-4

# --- VRAM check ---
if device.type == "cuda":
    vram_used = torch.cuda.max_memory_allocated() / 1e9
    print(f"\nPeak VRAM (forward, B=2, fp32): {vram_used:.2f} GB")
    print("Expected at training (B=4, bf16, grad-checkpoint, backward): ~30-38 GB")

print("\nSmoke test passed.")

## Phase G — Loss functions

Three losses combined per the paper:

```
L = L_det + 0.3 · L_cons + 0.01 · L_reg
```

- **`L_det`**: detection loss = CrossEntropy on `logits_4class` (ensemble output, primary signal).
  We also add auxiliary CE on `logits_video`, `logits_audio`, `logits_fused` with weight 0.3 to provide direct supervision to each branch of the hierarchical fusion. This is standard in deep-supervision setups.

- **`L_cons`**: cross-modal contrastive loss.
  In-batch positive pairs are `(v_embed_i, a_embed_i)`. Negatives are all other audios in the batch `(v_embed_i, a_embed_j) for j ≠ i`. Symmetric InfoNCE with temperature τ = 0.07.

- **`L_reg`**: L2 regularization on non-pretrained params (the consistency projector, fusion module, classifier heads). Already handled by AdamW's `weight_decay`; we expose it here for completeness and for paper-spec faithfulness.

In [ ]:
def contrastive_loss(v_embed: torch.Tensor, a_embed: torch.Tensor, tau: float = 0.07):
    """Symmetric InfoNCE between paired (v, a) embeddings.

    v_embed, a_embed: [B, D] L2-normalized.
    Returns scalar loss.
    """
    B = v_embed.size(0)
    logits = v_embed @ a_embed.T / tau            # [B, B] cosine similarities scaled
    targets = torch.arange(B, device=logits.device)
    loss_v2a = F.cross_entropy(logits, targets)     # video → audio
    loss_a2v = F.cross_entropy(logits.T, targets)   # audio → video
    return 0.5 * (loss_v2a + loss_a2v)


def detection_loss(out: dict, labels_4class: torch.Tensor,
                   aux_weight: float = 0.3):
    """Primary CE on ensemble logits + auxiliary CE on per-branch logits.

    NOTE: logits_4class is already log(softmax(...) avg), so we use NLL on it.
    The other three (fused/video/audio) are raw logits, so we use cross_entropy.
    """
    # Ensemble: log-probs already → NLL
    l_main = F.nll_loss(out["logits_4class"], labels_4class)
    # Auxiliary deep supervision
    l_v = F.cross_entropy(out["logits_video"], labels_4class)
    l_a = F.cross_entropy(out["logits_audio"], labels_4class)
    l_f = F.cross_entropy(out["logits_fused"], labels_4class)
    return l_main + aux_weight * (l_v + l_a + l_f) / 3.0


def total_loss(out: dict, labels_4class: torch.Tensor,
               w_cons: float = 0.3, tau: float = 0.07):
    """L = L_det + w_cons · L_cons.

    L_reg is handled by AdamW weight_decay in the train loop, so omitted here.
    """
    l_det  = detection_loss(out, labels_4class)
    l_cons = contrastive_loss(out["v_embed"], out["a_embed"], tau=tau)
    total = l_det + w_cons * l_cons
    return {
        "total":   total,
        "det":     l_det.detach(),
        "cons":    l_cons.detach(),
    }


# --- Smoke test losses on the forward pass output ---
labels_dev = labels.to(device)
with torch.no_grad():
    losses = total_loss(out, labels_dev)
print("Loss smoke test (random init, batch=2):")
for k, v in losses.items():
    print(f"  {k:<6}: {float(v):.4f}")

## Phase H — Save `AVXAIModel` to `models/avxai_model.py`

Write the full model code (classes only — no notebook driver) to a Python file so nb04 (training) can `from models.avxai_model import AVXAIModel, total_loss` cleanly.

In [ ]:
MODEL_PY_PATH = MODELS_DIR / "avxai_model.py"

source = '''"""AV-XAI model definition.

ViViT-B/16 (Kinetics-400 pretrained, full fine-tune)
+ WavLM-Large (LibriSpeech pretrained, full fine-tune)
+ Cross-modal consistency projection (256-dim L2-normalized)
+ Hierarchical fusion (3 levels: feature, decision, ensemble)
+ Binary head off fused features.

Generated by notebook 03. Imported by notebook 04 (training).
"""

import torch
import torch.nn as nn
import torch.nn.functional as F

from transformers import VivitConfig, VivitModel
from transformers import WavLMModel


VIVIT_MEAN = [0.5, 0.5, 0.5]
VIVIT_STD  = [0.5, 0.5, 0.5]


class ConsistencyProjector(nn.Module):
    """Projects video & audio embeddings to a shared L2-normalized space."""
    def __init__(self, d_video=768, d_audio=1024, d_shared=256):
        super().__init__()
        self.v_proj = nn.Sequential(
            nn.Linear(d_video, d_shared * 2),
            nn.GELU(),
            nn.Linear(d_shared * 2, d_shared),
        )
        self.a_proj = nn.Sequential(
            nn.Linear(d_audio, d_shared * 2),
            nn.GELU(),
            nn.Linear(d_shared * 2, d_shared),
        )

    def forward(self, vt, at):
        v = F.normalize(self.v_proj(vt), dim=-1)
        a = F.normalize(self.a_proj(at), dim=-1)
        return v, a


class HierarchicalFusion(nn.Module):
    """Three-level hierarchical fusion: feature + decision + ensemble."""
    def __init__(self, d_video=768, d_audio=1024, d_model=768, num_heads=4,
                 num_layers=2, n_classes=4, dropout=0.1):
        super().__init__()
        self.audio_align = nn.Linear(d_audio, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=num_heads,
            dim_feedforward=d_model * 4, dropout=dropout,
            batch_first=True, activation="gelu", norm_first=True,
        )
        self.fusion_transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.head_video  = nn.Linear(d_video, n_classes)
        self.head_audio  = nn.Linear(d_audio, n_classes)
        self.head_fused  = nn.Linear(d_model, n_classes)
        self.head_binary = nn.Linear(d_model, 2)

    def forward(self, vt, at):
        at_aligned = self.audio_align(at)
        tokens = torch.stack([vt, at_aligned], dim=1)
        tokens = self.fusion_transformer(tokens)
        fused = tokens.mean(dim=1)

        logits_video = self.head_video(vt)
        logits_audio = self.head_audio(at)
        logits_fused = self.head_fused(fused)

        probs_v = F.softmax(logits_video, dim=-1)
        probs_a = F.softmax(logits_audio, dim=-1)
        probs_f = F.softmax(logits_fused, dim=-1)
        ensemble_probs = (probs_v + probs_a + probs_f) / 3.0
        logits_4class  = torch.log(ensemble_probs + 1e-9)

        logits_binary = self.head_binary(fused)

        return {
            "logits_4class": logits_4class,
            "logits_binary": logits_binary,
            "logits_fused":  logits_fused,
            "logits_video":  logits_video,
            "logits_audio":  logits_audio,
            "fused":         fused,
        }


class AVXAIModel(nn.Module):
    def __init__(
        self,
        vivit_model_name: str = "google/vivit-b-16x2-kinetics400",
        wavlm_model_name: str = "microsoft/wavlm-large",
        num_frames: int = 16,
        d_shared: int = 256,
        n_classes: int = 4,
        fusion_layers: int = 2,
        fusion_heads: int = 4,
        use_gradient_checkpointing: bool = True,
    ):
        super().__init__()
        vivit_cfg = VivitConfig.from_pretrained(vivit_model_name)
        vivit_cfg.num_frames = num_frames
        self.vivit = VivitModel.from_pretrained(
            vivit_model_name, config=vivit_cfg, ignore_mismatched_sizes=True
        )
        self.wavlm = WavLMModel.from_pretrained(wavlm_model_name)

        self.consistency = ConsistencyProjector(
            d_video=self.vivit.config.hidden_size,
            d_audio=self.wavlm.config.hidden_size,
            d_shared=d_shared,
        )
        self.fusion = HierarchicalFusion(
            d_video=self.vivit.config.hidden_size,
            d_audio=self.wavlm.config.hidden_size,
            d_model=self.vivit.config.hidden_size,
            num_heads=fusion_heads, num_layers=fusion_layers, n_classes=n_classes,
        )

        self.register_buffer("vivit_mean", torch.tensor(VIVIT_MEAN).view(1, 1, 3, 1, 1), persistent=False)
        self.register_buffer("vivit_std",  torch.tensor(VIVIT_STD ).view(1, 1, 3, 1, 1), persistent=False)

        if use_gradient_checkpointing:
            self.vivit.gradient_checkpointing_enable()
            self.wavlm.gradient_checkpointing_enable()

    def _normalize_video(self, faces_uint8):
        x = faces_uint8.float() / 255.0
        return (x - self.vivit_mean) / self.vivit_std

    def _normalize_audio(self, audio_f32):
        m = audio_f32.mean(dim=-1, keepdim=True)
        s = audio_f32.std (dim=-1, keepdim=True) + 1e-7
        return (audio_f32 - m) / s

    def forward(self, faces, audio):
        if faces.dtype == torch.uint8:
            faces = self._normalize_video(faces)
        audio = self._normalize_audio(audio)

        vivit_out = self.vivit(pixel_values=faces)
        vt = vivit_out.last_hidden_state[:, 0]

        wavlm_out = self.wavlm(input_values=audio)
        at = wavlm_out.last_hidden_state.mean(dim=1)

        v_embed, a_embed = self.consistency(vt, at)
        fusion_out = self.fusion(vt, at)

        return {**fusion_out, "v_embed": v_embed, "a_embed": a_embed, "vt": vt, "at": at}


# ---------------------------------------------------------------------------
# Losses
# ---------------------------------------------------------------------------
def contrastive_loss(v_embed, a_embed, tau: float = 0.07):
    B = v_embed.size(0)
    logits  = v_embed @ a_embed.T / tau
    targets = torch.arange(B, device=logits.device)
    return 0.5 * (F.cross_entropy(logits, targets) + F.cross_entropy(logits.T, targets))


def detection_loss(out, labels_4class, aux_weight: float = 0.3):
    l_main = F.nll_loss(out["logits_4class"], labels_4class)
    l_v = F.cross_entropy(out["logits_video"], labels_4class)
    l_a = F.cross_entropy(out["logits_audio"], labels_4class)
    l_f = F.cross_entropy(out["logits_fused"], labels_4class)
    return l_main + aux_weight * (l_v + l_a + l_f) / 3.0


def total_loss(out, labels_4class, w_cons: float = 0.3, tau: float = 0.07):
    l_det  = detection_loss(out, labels_4class)
    l_cons = contrastive_loss(out["v_embed"], out["a_embed"], tau=tau)
    return {"total": l_det + w_cons * l_cons, "det": l_det.detach(), "cons": l_cons.detach()}
'''

MODEL_PY_PATH.write_text(source, encoding="utf-8")
print(f"Wrote: {MODEL_PY_PATH}")
print(f"Lines: {len(source.splitlines())}")

# Also create __init__.py so it's importable as `models.avxai_model`
init_fp = MODELS_DIR / "__init__.py"
if not init_fp.exists():
    init_fp.write_text("")
    print(f"Wrote: {init_fp}")

# Sanity-check import
print()
print("Testing import...")
sys.path.insert(0, str(PROJECT_ROOT))
from models.avxai_model import AVXAIModel as _AVXAIModel, total_loss as _total_loss
print(f"  imported AVXAIModel: {_AVXAIModel}")
print(f"  imported total_loss: {_total_loss}")

## Done.

**What's on disk now:**
```
models/
├── __init__.py
└── avxai_model.py        ← AVXAIModel, ConsistencyProjector, HierarchicalFusion, losses
```

**Verified:**
- ViViT-B/16 loads with 16-frame config (~86M params)
- WavLM-Large loads (~316M params)
- Forward pass on real preprocessed clips produces correct output shapes
- Consistency embeddings are L2-normalized
- Total params ≈ 405M, all trainable
- Losses compute cleanly

**Git commit:**
```cmd
git add notebooks/03_model_definition.ipynb models/avxai_model.py models/__init__.py
git commit -m "nb03: AVXAIModel definition — ViViT + WavLM + consistency + hierarchical fusion"
git push origin main
```

**Next: notebook 04 — training loop.**
- DataLoader for preprocessed `.pt` files (multi-worker, prefetch)
- AdamW with differential LR (1e-5 backbone / 5e-4 heads)
- Cosine schedule with 500 warmup steps
- bf16 autocast + gradient accumulation (effective batch 32)
- Per-epoch eval on val set (balanced accuracy, AUC)
- Checkpoint best model
- WandB or TensorBoard logging
- Estimated 8 epochs × ~45 min = ~6 hours